# Zero-copy clone and table metadata on AIDP Delta

Two Delta capabilities you can rely on in AIDP, demonstrated with explicit expected counts:

1. **Zero-copy clone** — `SHALLOW CLONE` creates a table that references the source's data files
   instead of copying them, and diverges from the source copy-on-write as soon as you write to it.
2. **Table and column metadata in SQL** — attaching and reading back key/value metadata via
   `TBLPROPERTIES`, column comments, and a registry table that scales to column scope.

## Prerequisites

- An AIDP cluster with Delta (tested on Spark 3.5.0 / Delta 3.2.0-oci-1.0.0).
- A catalog you can create schemas in. Set `CATALOG` below; the notebook creates a scratch schema
  and drops it at the end.

Every statement runs directly — there is no error-swallowing wrapper, so anything unsupported on your
build fails loudly at that cell rather than being recorded as a result.

In [ ]:
# Configuration -- point CATALOG at a catalog you can write to.
CATALOG = "default"
DB = "clone_metadata_demo"      # scratch schema; created here, dropped in the last cell
NS = f"{CATALOG}.{DB}"          # every statement below is fully qualified with this

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {NS}")
print(f"Working in {NS}")

In [ ]:
# A small table to clone and tag.
spark.sql(f"DROP TABLE IF EXISTS {NS}.customers")
spark.sql(f"""
    CREATE TABLE {NS}.customers (id INT, name STRING, email STRING, ssn STRING)
    USING delta
""")
spark.sql(f"""
    INSERT INTO {NS}.customers VALUES
        (1, 'Alice', 'alice@example.com', '111-22-3333'),
        (2, 'Bob',   'bob@example.com',   '444-55-6666')
""")
spark.table(f"{NS}.customers").show()

## 1. Zero-copy clone

`SHALLOW CLONE` copies only metadata: the new table's transaction log points at the **source's**
existing parquet files, so the clone is created in constant time regardless of table size. Writes to
either table create new files belonging to that table, leaving the other unchanged -- copy-on-write
divergence.

> **Operational warning.** A shallow clone depends on the source's data files. Running `VACUUM` on the
> source can delete files the clone still references and break it. Delta does not track that
> dependency for you, so either avoid vacuuming a cloned source or raise its retention window.
>
> If it has already happened, the repair is to re-clone -- but a bare
> `CREATE OR REPLACE TABLE ... SHALLOW CLONE` over a clone that holds its own data throws
> `DELTA_UNSUPPORTED_NON_EMPTY_CLONE`. `DELETE FROM` the clone first, then re-clone.

In [ ]:
# Create the clone. Constant time: no data is copied.
spark.sql(f"CREATE OR REPLACE TABLE {NS}.customers_shallow SHALLOW CLONE {NS}.customers")

clone_rows  = spark.table(f"{NS}.customers_shallow").count()
source_rows = spark.table(f"{NS}.customers").count()
print("rows in clone  (expect 2):", clone_rows)
print("rows in source (expect 2):", source_rows)
assert clone_rows == 2 and source_rows == 2, (clone_rows, source_rows)

In [ ]:
# Proving the clone is really zero-copy.
#
# DESCRIBE DETAIL alone does NOT prove it -- format/numFiles/sizeInBytes read the same
# for a deep copy, and DESCRIBE HISTORY reports operation = CLONE either way. The proof
# is in the CLONE commit's operationMetrics (numCopiedFiles = 0) and in the fact that the
# clone's data files ARE the source's files.
#
# `location` is deliberately not projected -- it embeds your object-storage namespace and
# bucket, which should not end up in a committed notebook output.
(spark.sql(f"DESCRIBE DETAIL {NS}.customers_shallow")
      .select("format", "numFiles", "sizeInBytes")
      .show(truncate=False))

clone_v0 = (spark.sql(f"DESCRIBE HISTORY {NS}.customers_shallow")
                 .orderBy("version").first())
print("first operation (expect CLONE):", clone_v0["operation"])
assert clone_v0["operation"] == "CLONE", clone_v0["operation"]

metrics = clone_v0["operationMetrics"] or {}
print("numCopiedFiles   (expect 0):", metrics.get("numCopiedFiles"))
print("sourceNumOfFiles (expect >0):", metrics.get("sourceNumOfFiles"))
assert int(metrics.get("numCopiedFiles", -1)) == 0, metrics

# The clone reads the source's own parquet files -- this is the zero-copy property.
clone_files  = set(spark.table(f"{NS}.customers_shallow").inputFiles())
source_files = set(spark.table(f"{NS}.customers").inputFiles())
print("clone reads exactly the source's files (expect True):", clone_files == source_files)
assert clone_files == source_files, (clone_files, source_files)

In [ ]:
# Copy-on-write divergence: write to the clone, source is untouched.
# The DELETE keeps the cell idempotent -- re-running it alone would otherwise
# accumulate rows and print 4 against "expect 3".
spark.sql(f"DELETE FROM {NS}.customers_shallow WHERE id = 99")
spark.sql(f"INSERT INTO {NS}.customers_shallow VALUES (99, 'Zoe', 'zoe@example.com', '999-88-7777')")

clone_rows  = spark.table(f"{NS}.customers_shallow").count()
source_rows = spark.table(f"{NS}.customers").count()
print("clone after insert  (expect 3):", clone_rows)
print("source unchanged    (expect 2):", source_rows)
assert clone_rows == 3 and source_rows == 2, (clone_rows, source_rows)

# The clone now has files of its own; the source's are still referenced.
print("clone no longer file-identical to source (expect True):",
      set(spark.table(f"{NS}.customers_shallow").inputFiles())
      != set(spark.table(f"{NS}.customers").inputFiles()))

### Deep clone

`DEEP CLONE` copies the data files as well, producing a fully independent table -- useful, but not
zero-copy, and it costs time proportional to table size.

It is not exercised here because availability differs by build: the open-source Delta 3.2 grammar
rejects `DEEP CLONE` (the OSS docs document shallow only), while Oracle's `3.2.0-oci` build may
accept it. Running it unconditionally would halt this notebook partway through on a build that does
not, so if you want it, add `CREATE OR REPLACE TABLE t DEEP CLONE customers` and confirm against
your own build.

## 2. Cloning structure without data

`CREATE TABLE ... AS SELECT ... WHERE 1=0` copies the schema of a query result, giving an empty table
with the same columns. The `USING delta` is explicit: without it the table is created with
`spark.sql.sources.default`, which is not Delta on a stock Spark configuration.

> `CREATE TABLE ... LIKE` is deliberately **not** used here. It is the v1 `CreateTableLikeCommand`, and
> both Spark 3.5 and Delta 3.2 resolve its source through the v1 `SessionCatalog` -- which only tracks
> the current database when the current catalog is `spark_catalog`. Against a plugin catalog it either
> fails with `TABLE_OR_VIEW_NOT_FOUND` or silently binds to a same-named table in `default`. Fully
> qualified three-part names avoid the whole class of problem, so this notebook uses them everywhere
> rather than relying on `USE`.

In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {NS}.customers_empty")
spark.sql(f"""
    CREATE TABLE {NS}.customers_empty
    USING delta
    AS SELECT * FROM {NS}.customers WHERE 1=0
""")

empty_rows = spark.table(f"{NS}.customers_empty").count()
empty_cols = len(spark.table(f"{NS}.customers_empty").columns)
print("structure-only rows    (expect 0):", empty_rows)
print("structure-only columns (expect 4):", empty_cols)
assert empty_rows == 0 and empty_cols == 4, (empty_rows, empty_cols)

## 3. Table and column metadata in SQL

Delta and Spark SQL let you **attach** metadata at three granularities. None of it is *enforced*:
there is no policy engine that reads a tag and redacts a column at query time, so if you want
enforcement you build it yourself -- typically a view that redacts the columns your registry marks
sensitive.

| Mechanism | Scope | Notes |
|---|---|---|
| `TBLPROPERTIES` | table | A real key/value store on the table. |
| Column `COMMENT` | column | The only per-column SQL slot; free text, so one dimension at best. |
| Registry table | column | Metadata as data you query and join. Scales across the lakehouse. |

In [ ]:
# Table-scoped key/value metadata.
# Note 'owner' is a reserved table property in Spark, so use a distinct key.
spark.sql(f"""
    ALTER TABLE {NS}.customers SET TBLPROPERTIES (
        'classification' = 'confidential',
        'data_owner'     = 'cdo',
        'pii'            = 'true'
    )
""")
props = {r["key"]: r["value"] for r in spark.sql(f"SHOW TBLPROPERTIES {NS}.customers").collect()}
spark.sql(f"SHOW TBLPROPERTIES {NS}.customers").show(truncate=False)
print("classification (expect 'confidential'):", props.get("classification"))
assert props.get("classification") == "confidential", props
assert props.get("data_owner") == "cdo" and props.get("pii") == "true", props

In [ ]:
# Column-scoped metadata via the comment field. Spark has no
# ALTER COLUMN ... SET TAGS, so the comment is the only per-column slot.
spark.sql(f"ALTER TABLE {NS}.customers ALTER COLUMN ssn COMMENT 'tag:pii;tag:masked'")

# DESCRIBE is run as a top-level statement, for the same parser reason as
# DESCRIBE DETAIL above.
desc = spark.sql(f"DESCRIBE {NS}.customers")
desc.show(truncate=False)

ssn_comment = {r["col_name"]: r["comment"] for r in desc.collect()}.get("ssn")
print("ssn comment (expect 'tag:pii;tag:masked'):", ssn_comment)
assert ssn_comment == "tag:pii;tag:masked", ssn_comment

### A registry table for column metadata

Putting the metadata in a table makes it queryable: one governance table, ordinary `INSERT`s, and
questions like "which columns across the lakehouse are marked PII?" become plain SQL.

In [ ]:
# Dropped first so the INSERT below is not additive when the cell is re-run.
spark.sql(f"DROP TABLE IF EXISTS {NS}.column_tags")
spark.sql(f"""
    CREATE TABLE {NS}.column_tags (
        catalog_name STRING, schema_name STRING, table_name STRING,
        column_name  STRING, tag_key     STRING, tag_value  STRING
    ) USING delta
""")

spark.sql(f"""
    INSERT INTO {NS}.column_tags VALUES
        ('{CATALOG}', '{DB}', 'customers', 'ssn',   'sensitivity', 'pii'),
        ('{CATALOG}', '{DB}', 'customers', 'email', 'sensitivity', 'pii')
""")

pii = spark.sql(f"""
    SELECT schema_name, table_name, column_name
    FROM {NS}.column_tags
    WHERE tag_key = 'sensitivity' AND tag_value = 'pii'
    ORDER BY column_name
""")
pii.show(truncate=False)
print("pii-tagged columns (expect 2):", pii.count())
assert pii.count() == 2, pii.count()
assert [r["column_name"] for r in pii.collect()] == ["email", "ssn"]

## Cleanup

Drops the scratch schema and everything in it. The shallow clone and its source go together, so
there is no vacuum-ordering concern here.

In [ ]:
spark.sql(f"DROP SCHEMA IF EXISTS {NS} CASCADE")
print(f"Dropped {NS}")